In [1]:
!pip install requests beautifulsoup4 arabic_reshaper python-bidi farasa nltk
!pip install --upgrade --force-reinstall pandas numpy # Add this line

  Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.1 MB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalli

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random

# Example Arabic news website (can be replaced with others)
urls = [
    'https://www.aljazeera.net/news/politics/',
    'https://www.alarabiya.net/arab-and-world'
]

def extract_text(url, count=5):
    headers = {"User-Agent": "Mozilla/5.0"}
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.content, 'html.parser')

    # Extract text data – this depends on the site structure
    articles = soup.find_all('p')[:count]
    return [' '.join(article.get_text().split()) for article in articles]

# Collect some text samples
texts = []
for url in urls:
    texts += extract_text(url, count=5)

# Assign a random score between 0 and 10 (you can also manually label them)
dataset = pd.DataFrame({
    'Text': texts,
    'Score': [round(random.uniform(0, 10), 1) for _ in range(len(texts))]
})
dataset.head()

,Text,Score
0,عثر على سجون كانت تابعة للواء الثامن بدرعا، ال...,2.1
1,أظهرت تصريحات إيال زامير حول نقص القوى البشرية...,2.1
2,نشرت صحيفة غارديان البريطانية تقريرا مطولا عن ...,8.6
3,بين مهمة استطلاعية ولجنة تقصي حقائق -وكلاهما آ...,6.2
4,شدد بيان صادر عن اجتماع أمير قطر الشيخ تميم بن...,0.2


In [5]:
import nltk
from nltk.corpus import stopwords
import re
from farasa.segmenter import FarasaSegmenter # Not used in this snippet, but imported
from farasa.pos import FarasaPOSTagger       # Not used in this snippet, but imported
from farasa.stemmer import FarasaStemmer
import pandas as pd # Make sure pandas is imported if running this cell standalone

# --- Download NLTK resources ---
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    print("Downloading missing 'punkt_tab' resource...")
    nltk.download('punkt_tab')

if 'stop_words' not in locals():
    stop_words = set(stopwords.words('arabic'))
if 'stemmer' not in locals():

    print("Initializing FarasaStemmer (interactive=True)...")
    stemmer = FarasaStemmer(interactive=True)
    print("FarasaStemmer initialized.")

def preprocess_text(text):
    if not isinstance(text, str):
      return ""

    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()


    try:
        tokens = nltk.word_tokenize(text)
    except Exception as e:
        print(f"Error tokenizing text: {text[:50]}... - Error: {e}")
        return ""


    tokens = [t for t in tokens if t not in stop_words]


    try:
        stemmed = [stemmer.stem(word) for word in tokens]
    except Exception as e:
        print(f"Error stemming tokens: {tokens[:10]}... - Error: {e}")

        stemmed = tokens

    return ' '.join(stemmed)

print("\nApplying preprocessing...")
dataset['Clean_Text'] = dataset['Text'].apply(preprocess_text)
print("Preprocessing finished.")

print("\nDataset Head with Clean_Text:")
print(dataset[['Text', 'Clean_Text', 'Score']].head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



Applying preprocessing...
Preprocessing finished.

Dataset Head with Clean_Text:
                                                Text  \
0  عثر على سجون كانت تابعة للواء الثامن بدرعا، ال...   
1  أظهرت تصريحات إيال زامير حول نقص القوى البشرية...   
2  نشرت صحيفة غارديان البريطانية تقريرا مطولا عن ...   
3  بين مهمة استطلاعية ولجنة تقصي حقائق -وكلاهما آ...   
4  شدد بيان صادر عن اجتماع أمير قطر الشيخ تميم بن...   

                                          Clean_Text  Score  
0  عثر سجن كان تابع لواء ثامن درع ، أعلن حل نفس ت...    2.1  
1  أظهر تصريح إيال زامير حول نقص قوة بشرية جيش إس...    2.1  
2  نشر صحيفة جارديان بريطاني تقرير مطول كاتب فلسط...    8.6  
3  مهمة استطلاعي لجنة تقصي حقيقة كلا آلي مراقبة د...    6.2  
4  شدد بيان صادر اجتماع أمير قطر شيخ تميم ابن حمد...    0.2  


In [7]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset['Clean_Text'])
sequences = tokenizer.texts_to_sequences(dataset['Clean_Text'])

# Padding
X = pad_sequences(sequences, maxlen=100)
y = dataset['Score'].values

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Function
def create_model(model_type='RNN'):
    model = tf.keras.Sequential()
    model.add(layers.Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=128, input_length=100))

    if model_type == 'RNN':
        model.add(layers.SimpleRNN(64))
    elif model_type == 'BiRNN':
        model.add(layers.Bidirectional(layers.SimpleRNN(64)))
    elif model_type == 'GRU':
        model.add(layers.GRU(64))
    elif model_type == 'LSTM':
        model.add(layers.LSTM(64))

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(1))  # Regression (score)

    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    return model

In [8]:
models_dict = {}
histories = {}

for mtype in ['RNN', 'BiRNN', 'GRU', 'LSTM']:
    print(f"Training {mtype} model...")
    model = create_model(model_type=mtype)
    history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=16)
    models_dict[mtype] = model
    histories[mtype] = history

Training RNN model...
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 28.1559 - mae: 4.1380 - val_loss: 3.9698 - val_mae: 1.9924
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - loss: 26.4479 - mae: 4.0177 - val_loss: 3.8208 - val_mae: 1.9547
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 24.0083 - mae: 3.8072 - val_loss: 3.3355 - val_mae: 1.8263
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - loss: 22.0543 - mae: 3.6035 - val_loss: 2.7213 - val_mae: 1.6496
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step - loss: 20.1341 - mae: 3.4061 - val_loss: 1.7998 - val_mae: 1.3416
Training BiRNN model...
Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - loss: 28.7033 - mae: 4.2133 - val_loss: 2.1586 - val_mae: 1.4692
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step - loss: 23.4028 - mae: 3.7511 - val_loss: 1.2876 - val_mae: 1.1347
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 19.4424 - mae: 3.3983 - val_loss: 0.5913 - val_mae: 0.7690
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - loss: 15.7905

In [11]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import pandas as pd # Ensure pandas is imported

min_rows_for_bleu = 2
num_samples_to_compare = 5

#----------------------------------------------------

for name, model in models_dict.items():
    # Make sure X_test is not empty
    if not hasattr(X_test, 'shape') or X_test.shape[0] == 0:
         print(f"\nSkipping {name} evaluation: X_test is empty or invalid.")
         continue


    try:
        preds = model.predict(X_test).flatten()
    except Exception as e:
        print(f"\nError during prediction for {name}: {e}")
        continue


    if not hasattr(y_test, '__len__') or len(y_test) == 0 or len(y_test) != len(preds):
        print(f"\nSkipping {name} evaluation: Mismatch in y_test/preds length or empty.")
        continue

    print(f"\n{name} Evaluation:")
    print(f"  MSE: {mean_squared_error(y_test, preds)}")
    print(f"  MAE: {mean_absolute_error(y_test, preds)}")

    # --- BLEU Score Calculation (Corrected) ---
    bleu_scores = []


    if 'dataset' in locals() and isinstance(dataset, pd.DataFrame) and 'Clean_Text' in dataset.columns:


        if len(dataset) >= min_rows_for_bleu and len(dataset) >= num_samples_to_compare + 1:

            num_pairs = min(num_samples_to_compare, len(dataset) - 1)

            print(f"  Calculating BLEU score for {num_pairs} pairs...")
            for i in range(num_pairs):

                ref_text = dataset['Clean_Text'].iloc[i]
                can_text = dataset['Clean_Text'].iloc[i+1]

                if isinstance(ref_text, str) and isinstance(can_text, str) and ref_text and can_text: # Make sure strings are not empty
                    reference = ref_text.split()
                    candidate = can_text.split()


                    if reference and candidate:
                        try:

                            score = sentence_bleu([reference], candidate, smoothing_function=SmoothingFunction().method1)
                            bleu_scores.append(score)
                        except Exception as e:
                            print(f"    Warning: Could not calculate BLEU for pair {i} vs {i+1}. Error: {e}")
                    else:
                        print(f"    Warning: Skipping BLEU for pair {i} vs {i+1} due to empty reference or candidate after split.")
                else:
                     print(f"    Warning: Skipping BLEU for pair {i} vs {i+1} due to non-string or empty Clean_Text.")

            if bleu_scores: # Avoid division by zero if no scores were calculated
                avg_bleu = sum(bleu_scores) / len(bleu_scores)
                print(f"  Avg BLEU (comparing row i vs i+1 for first {len(bleu_scores)} pairs): {avg_bleu:.4f}")
            else:
                print("  Avg BLEU: Not calculated (no valid pairs found).")
        else:
            print(f"  Avg BLEU: Not calculated (dataset size {len(dataset)} is less than required {max(min_rows_for_bleu, num_samples_to_compare + 1)} rows).")
    else:
        print("  Avg BLEU: Not calculated (dataset variable missing, not a DataFrame, or 'Clean_Text' column missing).")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step

RNN Evaluation:
  MSE: 1.7998134123270988
  MAE: 1.3415712475776673
  Avg BLEU: Not calculated (dataset size 5 is less than required 6 rows).
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 714ms/step

BiRNN Evaluation:
  MSE: 0.0014237060095274433
  MAE: 0.03773202896118155
  Avg BLEU: Not calculated (dataset size 5 is less than required 6 rows).
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 684ms/step

GRU Evaluation:
  MSE: 3.92712338001968
  MAE: 1.9816970959305764
  Avg BLEU: Not calculated (dataset size 5 is less than required 6 rows).
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 998ms/step

LSTM Evaluation:
  MSE: 4.204443141821942
  MAE: 2.0504738822579385
  Avg BLEU: Not calculated (dataset size 5 is less than required 6 rows).


In [12]:
dataset.to_csv("arabic_classification_dataset.csv", index=False)
